# PrimeVul Score Conversion

Converts lm-eval generate_until output into router training data.
Same structure as convert_dataset_7_model.ipynb.

Paper Eq. 1: s_i^(t) = (1/M) * sum_m evaluate(y_hat, y)  — with repeats=5, score = fraction correct.

In [1]:
import json
import os
import glob
import random
import pandas as pd
from collections import defaultdict

random.seed(42)

output_file_path = "../../datasets/primevul/balance_9models/gen"
os.makedirs(output_file_path, exist_ok=True)

# [org, model_name, output_dir_name]
# org/model_name  →  HuggingFace model ID used as key in scores dict
# output_dir_name →  directory name under output/primevul_gen/
model_list = [
    ["codellama",      "CodeLlama-7b-Instruct-hf",        "CodeLlama-7b-Instruct"],
    ["codellama",      "CodeLlama-13b-Instruct-hf",       "CodeLlama-13b-Instruct"],
    ["deepseek-ai",    "DeepSeek-Coder-V2-Lite-Instruct", "DeepSeek-Coder-V2-Lite-Instruct"],
    ["Qwen",           "Qwen2.5-Coder-14B-Instruct",      "Qwen2.5-Coder-14B-Instruct"],
    ["bigcode",        "starcoder2-15b-instruct-v0.1",    "starcoder2-15b-instruct"],
    ["Virtue-AI-HUB",  "VulnLLM-R-7B",                   "VulnLLM-R-7B"],
    ["google",         "codegemma-7b-it",                 "CodeGemma-7B-IT"],
    ["google",         "gemma-2-9b-it",                   "Gemma-2-9B-IT"],
    ["openai",         "gpt-4o-mini",                     "gpt-4o-mini"]
]

# PrimeVul (generate_until)

In [2]:
import re

def extract_answer(raw: str) -> str | None:
    """Mirror lm-eval 'extract-answer': pull Yes/No from \\boxed{...}."""
    m = re.search(r"\\boxed\{(Yes|No)", raw, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    m = re.search(r"\b(Yes|No)\b", raw, re.IGNORECASE)
    if m:
        return m.group(1).capitalize()
    return None   # malformed / unrecognised


# scores_by_idx[idx][model_id] = acc
# funcs_by_idx[idx]            = func text
scores_by_idx = {}
funcs_by_idx  = {}

for model_index, model_info in enumerate(model_list):
    model_pre  = model_info[0]
    model      = model_info[1]
    dir_name   = model_info[2]
    model_id   = model_pre + "/" + model

    pattern = f"../../output/primevul_gen_balance_9models/{dir_name}/*/samples_primevul_gen_*.jsonl"
    files = glob.glob(pattern)
    if not files:
        print(f"[skip] {model}: no samples file at {pattern}")
        continue

    n_docs = 0
    avg_acc = 0.0

    with open(files[0]) as f:
        for line in f:
            if not line.strip():
                continue
            entry  = json.loads(line)

            # Use doc["idx"] — the true PrimeVul dataset index — as the stable
            # key so scores align correctly even if doc_id differs across runs.
            idx    = entry["doc"]["idx"]
            resps  = entry.get("resps", [[]])[0]   # all M=5 raw responses
            target = entry["target"].strip()        # "Yes" or "No"

            # Paper Eq. 1: s = (1/M) * sum evaluate(y_hat_m, y)
            # Divide by len(resps)=M so malformed counts as wrong and
            # scores stay exact multiples of 0.2
            answers = [extract_answer(r) for r in resps]
            valid   = [a for a in answers if a is not None]
            correct = sum(1 for a in valid if a == target)
            acc     = correct / len(resps) if resps else 0.0

            # store score keyed by stable idx
            if idx not in scores_by_idx:
                scores_by_idx[idx] = {}
            scores_by_idx[idx][model_id] = acc

            # store func text (same across models, only needed once)
            if idx not in funcs_by_idx:
                funcs_by_idx[idx] = entry["doc"]["func"]

            n_docs  += 1
            avg_acc += acc

    avg_acc /= n_docs if n_docs else 1
    print(f"[ok] {model:<45} docs={n_docs}  avg_score={avg_acc:.3f}")

# Build output_data sorted by idx for reproducibility
output_data = [
    {"question": funcs_by_idx[idx], "scores": scores_by_idx[idx]}
    for idx in sorted(scores_by_idx.keys())
]

print(f"\nTotal docs: {len(output_data)}")

[ok] CodeLlama-7b-Instruct-hf                      docs=12008  avg_score=0.518
[ok] CodeLlama-13b-Instruct-hf                     docs=12008  avg_score=0.521
[ok] DeepSeek-Coder-V2-Lite-Instruct               docs=12008  avg_score=0.552
[ok] Qwen2.5-Coder-14B-Instruct                    docs=12008  avg_score=0.646
[ok] starcoder2-15b-instruct-v0.1                  docs=12008  avg_score=0.365
[ok] VulnLLM-R-7B                                  docs=12008  avg_score=0.656
[ok] codegemma-7b-it                               docs=12008  avg_score=0.515
[ok] gemma-2-9b-it                                 docs=12008  avg_score=0.586
[ok] gpt-4o-mini                                   docs=12008  avg_score=0.685

Total docs: 12008


In [3]:
# Train / test split  70 / 30  (same as paper)
train_split_index = random.sample(range(len(output_data)), len(output_data))
output_data = [output_data[idx] for idx in train_split_index]

train_split = output_data[:int(0.7 * len(output_data))]
test_split  = output_data[int(0.7 * len(output_data)):]

with open(os.path.join(output_file_path, "train.json"), "w") as f:
    json.dump(train_split, f)

with open(os.path.join(output_file_path, "test.json"), "w") as f:
    json.dump(test_split, f)

print(f"train: {len(train_split)}  test: {len(test_split)}")
print(f"Saved to {output_file_path}/")

train: 8405  test: 3603
Saved to ../../datasets/primevul/balance_9models/gen/


# Get ACC (per-model accuracy on full dataset)

In [4]:
with open(os.path.join(output_file_path, "train.json")) as f:
    train_data = json.load(f)
with open(os.path.join(output_file_path, "test.json")) as f:
    test_data = json.load(f)
output_data = train_data + test_data

# Build score dict from model list
correct_dict = {m[0]+"/"+m[1]: 0 for m in model_list}

data_size = len(output_data)
for item in output_data:
    for key, score in item["scores"].items():
        if key in correct_dict:
            correct_dict[key] += score / data_size * 100

df = pd.DataFrame.from_dict(correct_dict, orient="index", columns=["accuracy (%)"])

# Also show how many queries have partial scores (0 < s < 1) per model
partial_dict = {m[0]+"/"+m[1]: 0 for m in model_list}
for item in output_data:
    for key, score in item["scores"].items():
        if key in partial_dict and 1e-9 < score < 1 - 1e-9:
            partial_dict[key] += 1

df["partial_scores"] = pd.Series(partial_dict)
df["partial_%"] = (df["partial_scores"] / data_size * 100).round(1)
print(f"Total samples: {data_size} (train={len(train_data)}, test={len(test_data)})")
df

Total samples: 12008 (train=8405, test=3603)


,accuracy (%),partial_scores,partial_%
codellama/CodeLlama-7b-Instruct-hf,51.832112,4709,39.2
codellama/CodeLlama-13b-Instruct-hf,52.133578,3047,25.4
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct,55.183211,1528,12.7
Qwen/Qwen2.5-Coder-14B-Instruct,64.608594,3661,30.5
bigcode/starcoder2-15b-instruct-v0.1,36.475683,3726,31.0
Virtue-AI-HUB/VulnLLM-R-7B,65.626249,1827,15.2
google/codegemma-7b-it,51.490673,4923,41.0
google/gemma-2-9b-it,58.627582,426,3.5
openai/gpt-4o-mini,68.466023,354,2.9


In [5]:
with open(os.path.join(output_file_path, "train.json")) as f:
    train_data = json.load(f)
with open(os.path.join(output_file_path, "test.json")) as f:
    test_data = json.load(f)
output_data = train_data + test_data

M = 5   # number of runs per query

model_ids = list(output_data[0]["scores"].keys())
total_queries = len(output_data)

print(f"Total samples: {total_queries} (train={len(train_data)}, test={len(test_data)})")
print(f"{'Model':<55} {'correct':>10} {'total':>10} {'accuracy':>10}")
print("─" * 90)

for model_id in model_ids:
    # score = correct / M  →  correct = score * M
    total_correct = sum(item["scores"][model_id] * M for item in output_data)
    total_possible = total_queries * M
    accuracy = total_correct / total_possible * 100
    print(f"{model_id:<55} {total_correct:>10.0f} {total_possible:>10,} {accuracy:>9.2f}%")

Total samples: 12008 (train=8405, test=3603)
Model                                                      correct      total   accuracy
──────────────────────────────────────────────────────────────────────────────────────────
codellama/CodeLlama-7b-Instruct-hf                           31120     60,040     51.83%
codellama/CodeLlama-13b-Instruct-hf                          31301     60,040     52.13%
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct                  33132     60,040     55.18%
Qwen/Qwen2.5-Coder-14B-Instruct                              38791     60,040     64.61%
bigcode/starcoder2-15b-instruct-v0.1                         21900     60,040     36.48%
Virtue-AI-HUB/VulnLLM-R-7B                                   39402     60,040     65.63%
google/codegemma-7b-it                                       30915     60,040     51.49%
google/gemma-2-9b-it                                         35200     60,040     58.63%
openai/gpt-4o-mini                                           41